In [0]:
import urllib.request
import json

catalog = "dbr_dev"
schema  = "live_transit_monitor"
volume  = "project_volume"
batch_path = f"/Volumes/{catalog}/{schema}/{volume}/batch"
vehicles_file = f"{batch_path}/vehicles/vehicles.json"

vehicles_url = "https://files.cloudgdansk.pl/d/otwarte-dane/ztm/baza-pojazdow.json?v=2"

with urllib.request.urlopen(vehicles_url) as resp:
    payload = json.loads(resp.read().decode())

records = payload["results"]
print(f"Downloaded {len(records)} vehicle records.")

content = "\n".join(json.dumps(r, ensure_ascii=False) for r in records)
dbutils.fs.put(vehicles_file, content, overwrite=True)
print(f"Saved to {vehicles_file}")

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

vehicles_table = f"{catalog}.{schema}.bronze_vehicles"

df_vehicles = (
    spark.read
    .option("multiLine", "false")  
    .json(vehicles_file)
    
    .withColumn("source_file", F.lit("baza-pojazdow.json"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("load_date", F.current_date())
)

if not spark.catalog.tableExists(vehicles_table):
    df_vehicles.write.format("delta").saveAsTable(vehicles_table)
    print(f"Created {vehicles_table}")
else:
    delta_tbl = DeltaTable.forName(spark, vehicles_table)
    (delta_tbl.alias("t")
        .merge(df_vehicles.alias("s"), "t.vehicleCode = s.vehicleCode")
        .whenNotMatchedInsertAll()
        .execute())
    print(f"Merged into {vehicles_table}")

count = spark.table(vehicles_table).count()
print(f"{vehicles_table}: {count:,} rows")

In [0]:
display(spark.sql(f"""
    SELECT vehicleCode, transportationType, brand, model,
           productionYear, seats, standingPlaces, floorHeight,
           source_file, ingestion_timestamp
    FROM {vehicles_table}
    LIMIT 20
"""))